# 02 — From a set of galaxies to a first model

Notebooks 00 and 01 were about the data. This one builds something that predicts.

The obstacle is not that the problem is hard. It is that a catalog does not have the shape
machine learning usually expects. Almost every regressor you know takes a **fixed-length
vector**. A catalog is a **set**: a few hundred to a few thousand galaxies, a different number
in every simulation, in an order that carries no meaning.

So before anything else you have to answer one question:

> How do you turn a variable-size, unordered set of galaxies into a fixed number of numbers,
> without throwing away what you need?

Every model in this event is an answer to that question. This notebook builds the simplest
honest one, and it turns out not to be a straw man: it beats the graph network of notebook 04
on $\sigma_8$.

In [ ]:
# Setup. On Colab this installs the toolkit, mounts the data bucket and points the
# environment variables at it. Anywhere else -- a cluster with the data already on disk --
# it does nothing, which is why there is one set of notebooks rather than two.
import sys

if "google.colab" in sys.modules:
    # --force-reinstall, every time, on purpose. Installing only when the package is
    # missing means anyone who ran a notebook once keeps a stale copy forever, and during
    # an event where fixes are being pushed that is exactly backwards. --no-deps keeps it
    # to a few seconds: everything it depends on is already in the runtime.
    %pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/xwzhang98/kaai-robust-inference-hackathon-2026
    # Drop anything already imported from the old copy, so this works without a restart.
    for _name in [m for m in sys.modules if m.startswith("kaai_hackathon")]:
        del sys.modules[_name]

from kaai_hackathon.colab_setup import setup

setup()

In [ ]:
%matplotlib inline
import os
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from kaai_hackathon import PARAM_NAMES, PUBLIC_SUITES
from kaai_hackathon.catalog_io import read_catalog
from kaai_hackathon.progress import track
from kaai_hackathon.scoring import r2
from kaai_hackathon.splits import example_sims, load_labels, local_split

DATA_ROOT = Path(os.environ["CAMELS_HACKATHON_DATA"])
PARAMS_ROOT = Path(os.environ.get("CAMELS_HACKATHON_PARAMS", DATA_ROOT))
MSTAR_CUT = 1.3e-2          # M_star > 1.3e8 Msun/h, in the catalog's 1e10 Msun/h units
TARGETS = ("Omega_m", "sigma_8")


def catalog_path(suite, sim_id):
    return DATA_ROOT / suite / f"LH_{sim_id}" / "groups_090.hdf5"


def load(suite, sim_id):
    return read_catalog(catalog_path(suite, sim_id), group_fields=[],
                        subhalo_fields=["SubhaloPos", "SubhaloMassType"])

## 1. The shape of the problem

Look at what you are actually being handed.

In [ ]:
print(f"{'suite':14s} {'sim':>6s} {'subhalos':>10s} {'galaxies above the cut':>24s}")
for suite in PUBLIC_SUITES:
    for sim_id in example_sims(suite, 3):
        cat = load(suite, sim_id)
        mstar = cat.subhalo["SubhaloMassType"][:, 4]
        print(f"{suite:14s} {sim_id:6d} {cat.n_subhalos:10d} {int((mstar > MSTAR_CUT).sum()):24d}")

Different lengths, and not slightly different — the counts swing by factors of several between
simulations of the same suite. That variation is not a nuisance. **It is signal**: more matter
in the universe means more galaxies, so the count is already telling you something about
$\Omega_m$.

And there is no order. The rows come out of Subfind grouped by halo, but which galaxy is row 7
means nothing physical. Notebook 03 shows exactly what happens to a model that forgets this;
for now, take it as a constraint on what you are allowed to build.

Two rules follow, and everything else in this notebook is a consequence of them:

1. **Your summary must not depend on row order.** Sums, means, maxima, counts and histograms
   satisfy this. Indexing into rows does not.
2. **Your summary must have the same length for every catalog**, however many galaxies it has.

## 2. Three questions, three groups of numbers

A summary is a choice about what to keep. Here is a small one, built from three questions you
would ask about any population.

### How many are there?

In [ ]:
def counts(cat, cut=MSTAR_CUT):
    mstar = cat.subhalo["SubhaloMassType"][:, 4]
    return np.array([np.log1p(int((mstar > cut).sum())),
                     np.log1p(cat.n_subhalos),
                     np.log1p(cat.n_groups)])


N_FIT = 200
started = time.time()
rows, labels, suites = [], [], []
for suite in PUBLIC_SUITES:
    table = load_labels(PARAMS_ROOT, suite)
    for sim_id in track(local_split(suite)["train"][:N_FIT], f"{suite}: reading"):
        cat = load(suite, sim_id)
        rows.append(cat)                       # keep the catalogs; we reuse them below
        labels.append(table[sim_id, :2])
        suites.append(suite)
labels = np.asarray(labels)
suites = np.asarray(suites)
print(f"{len(rows)} catalogs loaded in {time.time() - started:.0f}s")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
n_gal = np.array([int((c.subhalo["SubhaloMassType"][:, 4] > MSTAR_CUT).sum()) for c in rows])
for j, (ax, target) in enumerate(zip(axes, TARGETS)):
    for suite in PUBLIC_SUITES:
        m = suites == suite
        ax.scatter(labels[m, j], n_gal[m], s=14, alpha=0.7, label=suite)
    ax.set_xlabel(target); ax.set_ylabel("galaxies above the cut"); ax.set_yscale("log")
axes[0].legend()
fig.suptitle("the plainest possible feature, against the two targets", y=1.02)
plt.tight_layout()

The left panel is close to a straight line and the right one is a cloud. One number, and it
already separates the two targets: **$\Omega_m$ is largely a counting problem, $\sigma_8$ is
not.** Keep that in mind for the rest of the event — it is the reason $\sigma_8$ stays hard for
every method here, and the reason a good $\Omega_m$ score should never be taken as evidence
that a model understands structure.

### How heavy are they?

The counts throw away every mass except through the cut. The **stellar mass function** — how
many galaxies at each mass — is the standard next step, and it is a histogram, so it is
order-independent and fixed-length by construction.

In [ ]:
MSTAR_BINS = np.linspace(-2.5, 2.0, 24)          # log10(M_star / 1e10 Msun/h)
CENTRES = 0.5 * (MSTAR_BINS[1:] + MSTAR_BINS[:-1])


def mass_function(cat, cut=MSTAR_CUT):
    mstar = cat.subhalo["SubhaloMassType"][:, 4]
    selected = mstar[mstar > cut]
    if not len(selected):
        return np.zeros(len(MSTAR_BINS) - 1)
    return np.histogram(np.log10(selected), bins=MSTAR_BINS)[0].astype(float)


fig, ax = plt.subplots(figsize=(7, 4.6))
order = np.argsort(labels[:, 0])
for rank, colour in ((0, "C0"), (len(order) // 2, "C2"), (len(order) - 1, "C3")):
    i = order[rank]
    ax.step(CENTRES, mass_function(rows[i]), where="mid", color=colour,
            label=f"Omega_m = {labels[i, 0]:.3f}  ({suites[i]})")
ax.set_yscale("log")
ax.set_xlabel(r"$\log_{10}(M_\star\,/\,10^{10} M_\odot/h)$")
ax.set_ylabel("galaxies per bin")
ax.legend(); plt.tight_layout()

The curves sit at different heights and bend differently. A histogram keeps that shape; a mean
mass would not.

### How clustered are they?

Neither of the first two knows *where* anything is. The cheapest measure of arrangement is the
mean number of neighbours within a few radii — and it must use the periodic minimum-image
convention, which `scipy.spatial.cKDTree(pos, boxsize=box)` does for you.

In [ ]:
from scipy.spatial import cKDTree

PAIR_RADII = (500.0, 1000.0, 2500.0, 5000.0)     # ckpc/h


def neighbours(cat, cut=MSTAR_CUT, radii=PAIR_RADII):
    mstar = cat.subhalo["SubhaloMassType"][:, 4]
    pos = cat.subhalo["SubhaloPos"][mstar > cut]
    if len(pos) < 2:
        return np.zeros(len(radii))
    wrapped = np.mod(pos.astype(np.float64), cat.box_size)
    tree = cKDTree(wrapped, boxsize=cat.box_size)
    # count_neighbors counts ordered pairs and includes each point with itself
    return (np.asarray(tree.count_neighbors(tree, list(radii))) - len(pos)) / len(pos)


pairs = np.array([neighbours(c) for c in rows])
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for j, (ax, target) in enumerate(zip(axes, TARGETS)):
    for suite in PUBLIC_SUITES:
        m = suites == suite
        ax.scatter(labels[m, j], pairs[m, 1], s=14, alpha=0.7, label=suite)
    ax.set_xlabel(target); ax.set_ylabel("mean neighbours within 1 cMpc/h")
    ax.set_yscale("log")
axes[0].legend(); plt.tight_layout()

## 3. Put them together

Three counts, two mass moments, 23 mass-function bins, four neighbour counts: 32 numbers,
whatever the catalog. That is exactly what the toolkit ships as `catalog_features`, so from
here on you can use that instead of the pieces.

In [ ]:
from kaai_hackathon.features import FEATURE_NAMES, N_FEATURES, catalog_features

started = time.time()
features = np.array([catalog_features(c) for c in rows])
print(f"{features.shape[0]} catalogs -> {features.shape[1]} features "
      f"in {time.time() - started:.0f}s")
print(f"\nthe {N_FEATURES} features:")
for i in range(0, N_FEATURES, 4):
    print("   " + "".join(f"{n:24s}" for n in FEATURE_NAMES[i:i + 4]))

## 4. Fit something

`RidgeCV` — linear regression with the regularization strength chosen by cross-validation.
Two deliberate choices worth copying into whatever you build next.

**Standardize the features.** They span counts, log masses and histogram bins, which live on
completely different scales, and a penalty that treats them as comparable would otherwise be
dominated by whichever happens to be largest.

**Split by simulation, not by catalog row.** Every galaxy in a simulation shares its
cosmology, so galaxies from one simulation are not independent samples. Splitting anywhere
below the simulation leaks the answer into your validation set and you will believe a score
that is not real.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

N_TEST = 60
test_features, test_labels, test_suites = [], [], []
for suite in PUBLIC_SUITES:
    table = load_labels(PARAMS_ROOT, suite)
    for sim_id in track(local_split(suite)["test"][:N_TEST], f"{suite}: held-out"):
        test_features.append(catalog_features(load(suite, sim_id)))
        test_labels.append(table[sim_id, :2])
        test_suites.append(suite)
test_features = np.asarray(test_features)
test_labels = np.asarray(test_labels)
test_suites = np.asarray(test_suites)


def fit_and_score(x_train, y_train, x_test, y_test, model=None):
    scaler = StandardScaler().fit(x_train)
    if model is None:
        model = RidgeCV(alphas=np.logspace(-3, 4, 30))
    model.fit(scaler.transform(x_train), y_train)
    prediction = model.predict(scaler.transform(x_test))
    return prediction, [r2(y_test[:, j], prediction[:, j]) for j in range(y_test.shape[1])]


prediction, scores = fit_and_score(features, labels, test_features, test_labels)
print(f"{'':16s} {'Omega_m':>9s} {'sigma_8':>9s}")
for suite in PUBLIC_SUITES:
    m = test_suites == suite
    print(f"{suite:16s} " + " ".join(
        f"{r2(test_labels[m, j], prediction[m, j]):9.3f}" for j in range(2)))
print(f"{'all together':16s} " + " ".join(f"{s:9.3f}" for s in scores))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for j, (ax, target, lo, hi) in enumerate(
        zip(axes, TARGETS, (0.1, 0.6), (0.5, 1.0))):
    for suite in PUBLIC_SUITES:
        m = test_suites == suite
        ax.scatter(test_labels[m, j], prediction[m, j], s=20, alpha=0.75, label=suite)
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.2, zorder=0)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
    ax.set_xlabel(f"true {target}"); ax.set_ylabel(f"predicted {target}")
    ax.set_title(f"{target}    $R^2$ = {scores[j]:+.3f}")
axes[0].legend(loc="upper left", fontsize=9)
plt.tight_layout()

Real numbers, from a linear model on 32 hand-picked features, in about a minute. That is your
floor. Anything you build should beat it, and if it does not, the interesting question is why.

## 5. Which numbers are doing the work?

More useful than the score is knowing where it comes from. Fit the same model on one group of
features at a time.

In [ ]:
GROUPS = {
    "counts only": [i for i, n in enumerate(FEATURE_NAMES) if n.startswith("log1p_n_")],
    "mass function only": [i for i, n in enumerate(FEATURE_NAMES)
                           if "mstar" in n and not n.startswith("log1p_n_")],
    "clustering only": [i for i, n in enumerate(FEATURE_NAMES) if "neighbours" in n],
    "everything": list(range(N_FEATURES)),
}

print(f"{'features':22s} {'how many':>9s} {'Omega_m':>9s} {'sigma_8':>9s}")
for name, index in GROUPS.items():
    _, group_scores = fit_and_score(features[:, index], labels,
                                    test_features[:, index], test_labels)
    print(f"{name:22s} {len(index):9d} " + " ".join(f"{s:9.3f}" for s in group_scores))

Read that table before going any further. It is the most informative thing in this notebook,
and it probably does not say what you expected.

**Three counts beat twenty-five mass-function numbers, on both targets.** Not by a little.
The plainest features in the whole vector — how many galaxies, how many subhalos, how many
halos — carry more than the entire shape of the mass function does.

That is worth sitting with, because it means **a good $\Omega_m$ score is weak evidence of
anything.** If most of it is available from counting, then beating this baseline on
$\Omega_m$ mostly demonstrates that you can count. The graph network in notebook 04 is handed
$\log_{10} N$ as an explicit input, deliberately, and so is de Santi's published model — the
count is treated as information rather than pretended away. Read every $\Omega_m$ number in
this event with that in mind, including ours.

**Four clustering numbers do more for $\sigma_8$ than twenty-five mass numbers do.** Per
feature it is the most efficient group here by a wide margin, and $\sigma_8$ is the parameter
that controls how clumpy matter is, so that is the physics behaving itself. It also says these
four numbers are leaving a lot on the table: four mean neighbour counts is an extremely coarse
description of a cosmic web. Notebook 04 replaces them with a graph over every nearby pair,
and the $\Omega_m$ score goes from 0.68 to 0.90.

**No group alone comes close to all of them together.** They are not measuring the same thing
three times; the information is genuinely spread across them. That is the ordinary situation,
and it is why an ablation is worth running before you decide what to throw away.

## 6. Does a nonlinear model help?

The features are hand-made and the model is linear. Relaxing the second is one line —
`HistGradientBoostingRegressor` is gradient-boosted trees, in scikit-learn, no extra install.
It fits one target at a time.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

started = time.time()
scaler = StandardScaler().fit(features)
boosted = []
for j in range(2):
    tree = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.06,
                                         early_stopping=True, random_state=0)
    tree.fit(scaler.transform(features), labels[:, j])
    boosted.append(tree.predict(scaler.transform(test_features)))
boosted = np.stack(boosted, axis=1)

print(f"{'model':22s} {'Omega_m':>9s} {'sigma_8':>9s}")
print(f"{'RidgeCV (linear)':22s} " + " ".join(f"{s:9.3f}" for s in scores))
print(f"{'gradient boosting':22s} " + " ".join(
    f"{r2(test_labels[:, j], boosted[:, j]):9.3f}" for j in range(2)))
print(f"\n[{time.time() - started:.0f}s]")

Boosting helps $\Omega_m$ and hurts $\sigma_8$ — worth knowing, and worth being suspicious of
on a single split, but not the point.

The point is the size of the move. Swapping a linear model for gradient-boosted trees on the
same 32 features changes $\Omega_m$ by a few hundredths. Swapping those 32 features for the
whole catalog as a graph, still with a fairly simple model on top, takes it from 0.68 to 0.90.

**The representation matters more than the regressor**, by an order of magnitude here. Time
spent deciding what to feed a model is generally better spent than time spent choosing the
model — which is the opposite of where most effort usually goes.

## 7. What this throws away, and where it goes next

Everything except stellar mass and position, for one thing — velocities never appear, so this
model cannot see the velocity field at all, and the `vel_noise` condition cannot touch it.
That is not robustness, it is blindness, and notebook 03 shows how to tell the two apart.

For another: after histogramming, a catalog whose galaxies sit in a tight filament and one
whose galaxies are scattered at random can produce **the same feature vector**. Only the four
neighbour counts see geometry at all, and they see it as four scalars.

Where the pieces stand at full scale, trained on all 900 public simulations per suite:

| | $\Omega_m$ | $\sigma_8$ |
|---|---|---|
| these 32 features + Ridge | 0.676 | **0.463** |
| the graph network, notebook 04 | **0.903** | 0.403 |

The graph network is far better on $\Omega_m$ and **loses on $\sigma_8$**. This baseline is not
a straw man, and the leaderboard is a table rather than a single number precisely because no
one method wins every column.

Next: **notebook 03** takes both of these apart under the shift conditions, and is where the
event's actual question starts.